# 04 · Dataset statistics for the cost model (|F1|, S2, ρ, P1)

Per dataset and support: transaction count D, item count N, mean/max/CV of transaction width,
exact frequent singletons |F1|, and the derived indicators used in the paper
(ρ = (|F1|²/2)/Σ|T|², P1 = (|F1|−1)/(d_avg(1+cv²))).

* `MODE = "replay"` (default): show the saved statistics
  (`results/campaign_2026-05/cost_stats_20261022.csv`, produced on 22 Oct 2026). Seconds.
* `MODE = "recompute"`: recompute from the shipped data files, compare with the saved file and
  write `results/derived/cost_stats.csv` (later notebooks then use it). Needs ~25 GB RAM for
  T25.D5M (125 M rows); restrict `DATASETS` on smaller machines.

The function below is `bench/exp_data.py::cost_stats`, copied verbatim.
**Note:** it drops duplicate (txn_id, item) rows first. The timing harness does not, which is why
Online Retail's mean width here (17.77) differs from the results table (18.24).

In [1]:
# ---- bundle configuration (same cell in every notebook) ----
import os, sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

BUNDLE = Path.cwd().resolve().parent          # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "baseline_runners.py").exists(), f"start Jupyter in <bundle>/notebooks (cwd={Path.cwd()})"
sys.path.insert(0, str(BUNDLE / "bench"))
DATA = Path(os.environ.get("FA_DATA_DIR", BUNDLE / "data"))
RESULTS = BUNDLE / "results"
DERIVED = RESULTS / "derived"                       # regenerated statistics, tables and figures

MAY = "campaign_2026-05"                           # the April/May 2026 GCE campaign
MAY_DIR = RESULTS / MAY
CAMPAIGN = os.environ.get("FA_CAMPAIGN", MAY) # set FA_CAMPAIGN=<id> to analyse a new run
CAMPAIGN_DIR = MAY_DIR if CAMPAIGN == MAY else RESULTS / "runs" / CAMPAIGN
AGG_CSV = (MAY_DIR / "results_rigorous_20260523_115442.csv" if CAMPAIGN == MAY
           else CAMPAIGN_DIR / f"results_rigorous_{CAMPAIGN}.csv")
LCM_CSV = MAY_DIR / "lcm_declat_results_20260523_055523.csv"
COST_CSV = (DERIVED / "cost_stats.csv") if (DERIVED / "cost_stats.csv").exists() else MAY_DIR / "cost_stats_20261022.csv"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(p):
    try:
        return Path(p).resolve().relative_to(BUNDLE).as_posix()
    except ValueError:
        return str(p)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"campaign : {CAMPAIGN}\nagg csv  : {rel(AGG_CSV)}")

campaign : campaign_2026-05
agg csv  : results/campaign_2026-05/results_rigorous_20260523_115442.csv


In [2]:
MODE = os.environ.get("FA_MODE", "replay")     # "replay" | "recompute"
DATASETS = None                                    # None = all 18, or a list of names
SUPPORTS = (1e-4, 5e-4, 1e-3, 5e-3, 1e-2)
SAVED = MAY_DIR / "cost_stats_20261022.csv"

In [3]:
def cost_stats(df: pd.DataFrame, supports=(1e-4, 5e-4, 1e-3, 5e-3, 1e-2)) -> list[dict]:
    """Cost-model statistics per support: |F1|, S2 (raw and after singleton
    filtering), rho = (|F1|^2/2)/S2, and the P1 ratio (|F1|-1)/(d_avg(1+cv^2))."""
    d = df[["txn_id", "item"]].drop_duplicates()
    sizes = d.groupby("txn_id").size()
    D = int(len(sizes))
    item_cnt = d.groupby("item").size()
    davg, sd, dmax = float(sizes.mean()), float(sizes.std(ddof=0)), int(sizes.max())
    cv = sd / davg if davg else float("nan")
    s2_raw = float((sizes.astype(np.float64) ** 2).sum())
    base = dict(D=D, N=int(len(item_cnt)), rows=int(len(d)), d_avg=davg,
                d_max=dmax, d_cv=cv, S2_raw=s2_raw)
    out = []
    for s in supports:
        mu = int(np.ceil(s * D)) if s > 0 else 1
        freq = item_cnt.index[item_cnt.to_numpy() >= mu]
        F1 = int(len(freq))
        fsizes = d[d["item"].isin(freq)].groupby("txn_id").size().astype(np.float64)
        s2_f = float((fsizes ** 2).sum())
        fdavg = float(fsizes.sum() / D)
        fcv = float(fsizes.reindex(sizes.index, fill_value=0).std(ddof=0) / fdavg) if fdavg else float("nan")
        out.append(dict(base, min_support=s, min_count=mu, F1=F1, S2_filtered=s2_f,
                        rho_raw=(F1 ** 2 / 2) / s2_raw if s2_raw else float("nan"),
                        rho_filtered=(F1 ** 2 / 2) / s2_f if s2_f else float("nan"),
                        P1_raw=(F1 - 1) / (davg * (1 + cv ** 2)) if davg else float("nan"),
                        P1_filtered=(F1 - 1) / (fdavg * (1 + fcv ** 2)) if fdavg else float("nan")))
    return out

In [4]:
saved = pd.read_csv(SAVED)
if MODE == "replay":
    print(f"replay: {rel(SAVED)} (sha256 {sha256(SAVED)[:16]}…), {saved.dataset.nunique()} datasets")
    display(saved[np.isclose(saved.min_support, 1e-4)][["dataset", "D", "N", "F1", "d_avg", "d_max", "d_cv", "rho_raw", "P1_raw"]])

replay: results/campaign_2026-05/cost_stats_20261022.csv (sha256 5aa8c28db2814c67…), 18 datasets


,dataset,D,N,F1,d_avg,d_max,d_cv,rho_raw,P1_raw
0,Groceries,9835,169,169,4.409456,32,0.813978,0.044918,22.916417
5,T12.I4.D5k.N1k.L1k,5000,981,981,11.970400,27,0.285567,0.620976,75.695735
10,T5.I2.D25k.N1k.L1k,24951,900,887,4.993387,16,0.445313,0.527682,148.071539
15,BMS-WebView-1,59602,497,451,2.510637,267,1.932889,0.057158,37.845244
20,T3.I2.D100k.N50k.L50k,99384,43856,11121,3.057917,14,0.541927,51.435321,2810.934262
25,BMS-WebView-2,77512,3340,2984,4.622226,161,1.313449,0.986521,236.816579
30,Online Retail,28816,4632,4062,17.772279,674,1.923310,0.192892,48.626526
35,T15.I8.D50k.N1k.L1k.high,50000,807,805,14.983880,33,0.257976,0.027062,50.309479
40,T15.I8.D50k.N1k.L1k.low,50000,1000,1000,15.006180,34,0.258167,0.041633,62.412748
45,Retail (Belgian),88162,16470,8919,10.305755,76,0.792017,2.610333,531.768061


In [5]:
if MODE == "recompute":
    from load_datasets import ALL_DATASETS
    names = DATASETS or list(saved.dataset.unique())
    rows = []
    for n in names:
        t0 = time.time()
        df = ALL_DATASETS[n]()
        rows += [dict(dataset=n, **r) for r in cost_stats(df, SUPPORTS)]
        print(f"{n:<28} {len(df):>12,} rows  {time.time() - t0:6.1f}s")
        del df
    fresh = pd.DataFrame(rows)
    DERIVED.mkdir(parents=True, exist_ok=True)
    out = DERIVED / "cost_stats.csv"
    if DATASETS is None:
        fresh.to_csv(out, index=False); print("wrote", rel(out))
    # compare with the saved file
    key = ["dataset", "min_support"]
    m = fresh.merge(saved, on=key, suffixes=("", "_saved"))
    worst = {}
    for c in ["D", "N", "rows", "d_avg", "d_max", "d_cv", "F1", "S2_raw", "S2_filtered", "rho_raw", "P1_raw", "P1_filtered"]:
        r = (m[c] - m[f"{c}_saved"]).abs() / m[f"{c}_saved"].abs().clip(lower=1e-12)
        worst[c] = float(r.max())
    print("max relative difference vs saved:", worst)
    assert max(worst.values()) < 1e-9, "recomputed statistics differ from the saved file"
    print("RECOMPUTED STATISTICS MATCH the saved file for", m.dataset.nunique(), "datasets")